In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch

def draw_equal_quincunx_with_arrows(radius=1.0, arrow_len=None, savepath=None):
    """
    Five equal circles (radius = radius) in quincunx.
    Outer four are tangent to the center circle (centers at distance d=2*radius).
    Dashed guide circle goes through the outer centers.
    Arrows start at the CENTER of each outer circle:
        top & bottom: upwards; left & right: downwards.
    The dashed guide is rendered BEHIND the disks (hidden where overlapped).
    """
    s = float(radius)
    if s <= 0:
        raise ValueError("radius must be positive.")

    d = 2.0 * s
    if arrow_len is None:
        arrow_len = 1.1 * s

    centers = {
        "top":    (0.0,  d),
        "right":  ( d,  0.0),
        "bottom": (0.0, -d),
        "left":   (-d,  0.0),
        "center": (0.0,  0.0),
    }

    # z-order: guide (1) < circles (3) < arrows (4)
    z_guide, z_circles, z_arrows = 1, 3, 4

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_aspect('equal')
    ax.axis('off')

    # --- dashed guide circle FIRST (behind) ---
    ax.add_patch(Circle((0.0, 0.0), d, ec="#4a5568", fc="none",
                        lw=1.8, ls="--", zorder=z_guide))

    # --- disks on top of guide (opaque facecolors hide the dashed where overlapped) ---
    for key in ["top", "right", "bottom", "left", "center"]:
        cx, cy = centers[key]
        ax.add_patch(Circle((cx, cy), s, ec="#2b6cb0", fc="#e6eefc",
                            lw=2, zorder=z_circles))

    # --- arrows on top ---
    def vertical_arrow_from_center(cx, cy, direction):
        if direction == "up":
            start, end = (cx, cy), (cx, cy + arrow_len)
        elif direction == "down":
            start, end = (cx, cy), (cx, cy - arrow_len)
        else:
            raise ValueError("direction must be 'up' or 'down'")
        a = FancyArrowPatch(
            start, end, arrowstyle='-|>', mutation_scale=16,
            lw=2.0, color='#44515c', zorder=z_arrows
        )
        ax.add_patch(a)

    vertical_arrow_from_center(*centers["top"],    "up")
    vertical_arrow_from_center(*centers["bottom"], "up")
    vertical_arrow_from_center(*centers["left"],   "down")
    vertical_arrow_from_center(*centers["right"],  "down")

    # limits include arrows
    pad = 0.3 * s
    x_pad = s + pad
    y_pad = max(s, arrow_len) + pad
    ax.set_xlim(-d - x_pad, d + x_pad)
    ax.set_ylim(-d - y_pad, d + y_pad)

    if savepath:
        fig.savefig(savepath, dpi=300, bbox_inches='tight', pad_inches=0.1)
    plt.show()
    plt.close(fig)

# Example:
draw_equal_quincunx_with_arrows(radius=1.0, arrow_len=1.1, savepath='fig1.png')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def animate_two_circles_with_roulette_and_arrow(radius=1.0, frames=360, interval=25,
                                                clockwise=False, savepath=None, fps=30, dpi=150):
    R = float(radius); r = R
    sgn = -1.0 if clockwise else 1.0
    phi0 = np.pi / 2
    phi  = phi0 + sgn * np.linspace(0.0, 2*np.pi, frames, endpoint=False)
    psi  = 2.0 * (phi - phi0)           # external rolling, R=r -> factor 2
    cx   = (R + r) * np.cos(phi)
    cy   = (R + r) * np.sin(phi)
    alpha0 = phi0 + np.pi               # marker initially toward origin
    Px = cx + r * np.cos(psi + alpha0)
    Py = cy + r * np.sin(psi + alpha0)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_aspect('equal'); ax.axis('off')
    center_disk  = Circle((0, 0), R, ec="#2b6cb0", fc="#e6eefc", lw=2, zorder=3)
    rolling_disk = Circle((cx[0], cy[0]), r, ec="#d65f5f", fc="#fdecec", lw=2, zorder=3)
    ax.add_patch(center_disk); ax.add_patch(rolling_disk)
    path_line, = ax.plot([], [], '-', lw=2.0, color='red', zorder=2)
    P_dot,     = ax.plot(Px[0], Py[0], 'o', ms=6, mfc='white', mec='#222', zorder=4)
    arrow = FancyArrowPatch((cx[0], cy[0]), (Px[0], Py[0]),
                            arrowstyle='-|>', mutation_scale=16, lw=2.0,
                            color='#44515c', zorder=4)
    ax.add_patch(arrow)

    pad = 0.4 * R
    ax.set_xlim(-3*R - pad, 3*R + pad)
    ax.set_ylim(-3*R - pad, 3*R + pad)

    def update(i):
        rolling_disk.center = (cx[i], cy[i])
        P_dot.set_data(Px[i], Py[i])
        path_line.set_data(Px[:i+1], Py[:i+1])
        ang = psi[i] + alpha0
        arr_len = 0.9 * R
        start = (cx[i], cy[i])
        end   = (cx[i] + arr_len*np.cos(ang), cy[i] + arr_len*np.sin(ang))
        arrow.set_positions(start, end)
        return rolling_disk, P_dot, path_line, arrow

    anim = FuncAnimation(fig, update, frames=frames, interval=interval, blit=True)

    # ---- Save if requested ----
    if savepath is not None:
        ext = savepath.split(".")[-1].lower()
        if ext in {"mp4", "m4v", "mov"}:
            from matplotlib.animation import FFMpegWriter
            anim.save(savepath, writer=FFMpegWriter(fps=fps), dpi=dpi)
        elif ext in {"gif"}:
            from matplotlib.animation import PillowWriter
            anim.save(savepath, writer=PillowWriter(fps=fps), dpi=dpi)
        else:
            raise ValueError("Unsupported extension; use .mp4 or .gif")

    plt.close(fig)   # close figure to avoid duplicate display in notebooks
    return HTML(anim.to_jshtml())

# Show inline only
animate_two_circles_with_roulette_and_arrow(radius=1.0)
# Save as MP4 (needs ffmpeg)
animate_two_circles_with_roulette_and_arrow(radius=1.0, savepath="roll.mp4", fps=30, dpi=150)
# Save as GIF (uses Pillow)
animate_two_circles_with_roulette_and_arrow(radius=1.0, savepath="roll.gif", fps=20)